In [ ]:
import os
    
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Rectangle
from matplotlib import animation
from IPython.display import HTML
import seaborn as sns
from PIL import Image

from tqdm import tqdm
from scipy.spatial.distance import pdist, cdist, squareform
from scipy.stats import pearsonr, spearmanr, entropy, rankdata
from sklearn.manifold import MDS
from scipy.ndimage import gaussian_filter1d

import manifold_dynamics.tuning_utils as tut

RAND = 0
RESP = (50,220)
BASE = (-50,0)
ONSET = 50
RESP = slice(ONSET + RESP[0], ONSET + RESP[1])
BASE = slice(ONSET + BASE[0], ONSET + BASE[1])

In [ ]:
DATA_DIR = '../../datasets/NNN/'
IMG_DIR = '../../datasets/NNN/NSD1000_LOC'

CAT = 'face'
dat = pd.read_pickle(os.path.join(DATA_DIR, (f'{CAT}_roi_data.pkl')))
ROI_LIST = list(dat['roi'].unique())
print(f'Unique face ROIs: {ROI_LIST}')

with open(f'../../datasets/NNN/{CAT}_mins.pkl', 'rb') as f:
    mins = pickle.load(f)
    
SCALES = {k: v[0] for k,v in mins.items()}

SAVE_DIR = '../../../buckets/manifold-dynamics/time-time/'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

In [ ]:
X = tut.trial_averaged_psth(dat, ROI)  # (units, time, images)

# z-score across images for each (unit, time)
mu = np.nanmean(X, axis=2, keepdims=True)
sd = np.nanstd(X, axis=2, keepdims=True) + 1e-8
Xz = (X - mu) / sd

order = tut.rank_images_by_response(X)      # IMPORTANT: rank on the original X (or on split A if CV)
scale = SCALES[ROI]
idx = order[:scale]

Rt, _ = tut.tuning_rdm(X=Xz, indices=idx, tstart=100, tend=300, metric='correlation')
print(tut.ED2(Rt))
Rt, _ = tut.tuning_rdm(X=Xz, indices=idx, tstart=0, tend=50, metric='correlation')
print(tut.ED2(Rt))

order = tut.rank_images_by_response(X)      # IMPORTANT: rank on the original X (or on split A if CV)
idx = np.random.choice(order, scale)

Rt, _ = tut.tuning_rdm(X=Xz, indices=idx, tstart=100, tend=300, metric='correlation')
print(tut.ED2(Rt))
Rt, _ = tut.tuning_rdm(X=Xz, indices=idx, tstart=0, tend=50, metric='correlation')
print(tut.ED2(Rt))

# X = tut.trial_averaged_psth(dat, ROI)
# # normalize each image vector (over units) to unit length at each time
# norm = np.linalg.norm(X, axis=0, keepdims=True) + 1e-8   # norm over units -> shape (1,time,images)
# Xn = X / norm
# Rt, _ = tut.tuning_rdm(X=Xn, indices=idx, tstart=100, tend=300, metric='correlation')
# print(tut.ED2(Rt))

In [ ]:
ROI = 'Unknown_19_F' # Unknown_19_F, MF1_7_F, MF1_8_F, MF1_9_F, AF3_18_F, MB1_3_B
MODE = 'top'
scale = SCALES[ROI]
# 100 t0, 200 window for medial
# 150 to, 300 window for anterior
t0 = 100
window = 200 #

Rt, _ = tut.static_rdm(dat, ROI, mode=MODE, scale=scale,
                  tstart=t0, tend=t0 + window)

print(tut.ED2(Rt))
# np.fill_diagonal(Rt, np.nan)

Ra, _ = tut.static_rdm(dat, ROI, mode=MODE, scale=1072,
                  tstart=t0, tend=t0 + window)
print(tut.ED2(Ra))

# Rss = []
# for i in tqdm(range(50)):
#     Rs, _ = tut.static_rdm(dat, ROI, mode='shuff', scale=scale,
#                       tstart=t0, tend=t0 + window, random_state=i)
#     Rss.append(Rs)
# Rs_EDs = np.array([tut.ED2(Rs[t0:window, t0:window]) for Rs in Rss])
# print(np.nanmean(Rs_EDs))